# Step 6 — manual stop additions

Stations added by hand on top of step 5's night train stops, so the catalog
covers places no night train serves *today* but a target network would: large
urban areas without a qualified stop, tourism regions, and major ferry hubs.

**This notebook is the source of truth for those additions.** The selections
live in the `ADDITIONS` cells below, grouped by region and keyed by OSM stop
id, and everything else (name, coordinates, country) is looked up from step 3b
and step 4 at run time — so a stop is added or removed by editing one line
here, and `git diff` shows exactly what changed and why.

## The `reason` field

Every addition carries a free-text `reason`. The design requires that *"why is
station X (not) included?"* be answerable from the data alone, including by
people outside the project, and today it is not: the original step 6 selection
was made manually with no record of the criteria, so these 398 stops arrive
with `reason` empty.

Filling them in is Johanna's — they are her selections. Suggested vocabulary,
kept short and greppable:

| reason | when |
|---|---|
| `fua:<city>` | functional urban area with no qualified stop |
| `tourism:<region>` | tourism destination |
| `ferry:<port>` | major ferry hub |
| `border` | border/interchange station |
| `network` | needed to make a corridor coherent |

Anything unfilled is reported at the bottom, so the gap stays visible rather
than quietly becoming permanent.

## Output

`data/step6_manual_additions.csv` — `stop_id, stop_name, country, stop_lat,
stop_lon, reason` — consumed by `step7_export_seed_stops.py`, which unions it
with the current step 5 output.

The legacy `step6_metropol.csv` mixed these additions together with an earlier
step 5 run and had no reason column; it is kept only as the source this
notebook was bootstrapped from.

In [1]:
import csv
from collections import Counter

from data_sources import DATA_DIR, ensure_local

OUTPUT_PATH = DATA_DIR / "step6_manual_additions.csv"

## The additions

One dict per region, `stop_id: (name, reason)`. The name is a comment for
readability only — it is re-read from step 3b when the file is written, so a
stale name here cannot corrupt the output. Add a stop by adding a line; remove
one by deleting its line.

In [2]:
# Germany, Austria, Switzerland
ADDITIONS_GERMANY = {
    # --- DE ---
    "osm:n31485922": ("Bayreuth Hbf", ""),
    "osm:n1874501382": ("Bielefeld Hauptbahnhof", ""),
    "osm:w24806780": ("Braunschweig Hauptbahnhof", ""),
    "osm:n26562398": ("Bremerhaven Hauptbahnhof", ""),
    "osm:n2711388096": ("Böblingen", ""),
    "osm:n3607858763": ("Chemnitz Hauptbahnhof", ""),
    "osm:n2599505466": ("Cottbus Hauptbahnhof / Chóśebuz głowne dwórnišćo", ""),
    "osm:n3175310444": ("Darmstadt Hauptbahnhof", ""),
    "osm:n4189000814": ("Flensburg / Flensborg", ""),
    "osm:n1840958277": ("Gera Hauptbahnhof", ""),
    "osm:n2461322542": ("Gesundbrunnen", ""),
    "osm:n4349314485": ("Göppingen", ""),
    "osm:n1438696887": ("Görlitz", ""),
    "osm:n3450444902": ("Gütersloh Hbf", ""),
    "osm:n3135336139": ("Harburg", ""),
    "osm:n767185754": ("Hauptbahnhof", ""),
    "osm:n11069371580": ("Hauptbahnhof (Arnulf-Klett-Platz)", ""),
    "osm:n2820941790": ("Heidelberg Hauptbahnhof", ""),
    "osm:n27385328": ("Heilbronn Hauptbahnhof", ""),
    "osm:n3616040153": ("Hildesheim Hauptbahnhof", ""),
    "osm:n3543804400": ("Ingolstadt Hbf", ""),
    "osm:n1126168394": ("Jena Paradies", ""),
    "osm:n30959690": ("Kaiserslautern Hauptbahnhof", ""),
    "osm:n4530820004": ("Kempten (Allgäu) Hbf", ""),
    "osm:n4257641280": ("Kiel Hauptbahnhof", ""),
    "osm:n534753716": ("Landshut (Bay) Hbf", ""),
    "osm:n3087634633": ("Magdeburg Hauptbahnhof", ""),
    "osm:n7160009313": ("Neumünster", ""),
    "osm:n91753264": ("Oldenburg (Oldb) Hbf", ""),
    "osm:n268894281": ("Osnabrück Hauptbahnhof", ""),
    "osm:n2837556546": ("Ostbahnhof", ""),
    "osm:n2675283037": ("Paderborn Hauptbahnhof", ""),
    "osm:n25972727": ("Pforzheim Hauptbahnhof", ""),
    "osm:n1755712810": ("Plauen (Vogtl) ob Bf", ""),
    "osm:n4565521156": ("Remscheid-Lennep", ""),
    "osm:n25233549": ("Reutlingen Hbf", ""),
    "osm:n987773654": ("Rostock Hauptbahnhof", ""),
    "osm:n259449966": ("Saarbrücken Hauptbahnhof", ""),
    "osm:n27381920": ("Schweinfurt Hbf", ""),
    "osm:n252098248": ("Schwerin Hauptbahnhof", ""),
    "osm:n4208739688": ("Spandau", ""),
    "osm:n277350630": ("Stralsund Hbf", ""),
    "osm:n7103322725": ("Südkreuz (Nord-Süd)", ""),
    "osm:n745097775": ("Trier Hbf", ""),
    "osm:n338899629": ("Wolfsburg Hauptbahnhof", ""),
    # --- CH ---
    "osm:n310603375": ("Lugano Stazione (funicolare)", ""),
    "osm:n2051794005": ("Luzern", ""),
    "osm:n2428167137": ("Schaffhausen", ""),
    "osm:n3080746026": ("Schönenwerd", ""),
    "osm:n1346888802": ("St. Gallen", ""),
    "osm:n3081154442": ("Thun", ""),
    "osm:n1286602751": ("Winterthur", ""),
    "osm:n60093107": ("Wien Westbahnhof", "network — major Vienna terminus, referenced by route fixtures"),
}

In [3]:
# France, Benelux
ADDITIONS_FRANCE = {
    # --- FR ---
    "osm:n4290854846": ("Aix-en-Provence", ""),
    "osm:n1680885216": ("Amiens", ""),
    "osm:n3486353293": ("Angers Saint-Laud", ""),
    "osm:n5061961433": ("Annecy", ""),
    "osm:n7167504997": ("Arras", ""),
    "osm:n3805976209": ("Avignon-Centre", ""),
    "osm:n2501252269": ("Belfort", ""),
    "osm:n2500070617": ("Besançon-Viotte", ""),
    "osm:n8213648712": ("Boulogne Ville", ""),
    "osm:n194212267": ("Bourges", ""),
    "osm:n2207570062": ("Brest", ""),
    "osm:w73575850": ("Briançon", ""),
    "osm:n5598384401": ("Caen", ""),
    "osm:n9183304884": ("Calais-Ville", ""),
    "osm:n5066478129": ("Chambéry - Challes-les-Eaux", ""),
    "osm:n8303862255": ("Chartres", ""),
    "osm:w112095036": ("Cherbourg", ""),
    "osm:n10936459654": ("Clermont-Ferrand", ""),
    "osm:n398836628": ("Colmar", ""),
    "osm:n11556100824": ("Douai", ""),
    "osm:n312805118": ("Dunkerque", ""),
    "osm:n5070332503": ("Grenoble", ""),
    "osm:n4975049664": ("La Rochelle", ""),
    "osm:n847701543": ("Le Havre", ""),
    "osm:n8745537419": ("Lille-Flandres", ""),
    "osm:n3471044194": ("Limoges-Bénédictins", ""),
    "osm:n10940619366": ("Lorient", ""),
    "osm:n4290857018": ("Lyon Perrache", ""),
    "osm:n11225690169": ("Martigues", ""),
    "osm:n4290857026": ("Metz", ""),
    "osm:n4225150278": ("Montbéliard", ""),
    "osm:n2502268309": ("Mulhouse-Ville", ""),
    "osm:n4290857032": ("Nancy", ""),
    "osm:n3486337796": ("Nantes", ""),
    "osm:n9912502487": ("Poitiers", ""),
    "osm:n4986753876": ("Quimper", ""),
    "osm:n2517400258": ("Reims", ""),
    "osm:n4250849558": ("Rennes", ""),
    "osm:n2483753165": ("Roanne", ""),
    "osm:n2076751841": ("Rouen Rive-Droite", ""),
    "osm:n4280767167": ("Saint-Brieuc", ""),
    "osm:n829527258": ("Saint-Raphaël-Valescure", ""),
    "osm:n2010251922": ("Saint-Étienne Châteaucreux", ""),
    "osm:n3069229440": ("Strasbourg", ""),
    "osm:n2506173917": ("Troyes", ""),
    "osm:n10935384072": ("Valence-Ville", ""),
    "osm:n1648968005": ("Valenciennes", ""),
    "osm:n394710073": ("Vannes", ""),
    # --- BE ---
    "osm:n2929614444": ("Brugge", ""),
    "osm:n12095463891": ("Diamant", ""),
    "osm:n4878787366": ("Gare du Midi - Zuidstation", ""),
    "osm:n1178257779": ("Gent-Sint-Pieters", ""),
    "osm:n21309047": ("Kortrijk", ""),
    "osm:n446059037": ("La Louvière-Centre", ""),
    "osm:n7261826908": ("Mechelen-Nekkerspoel", ""),
    "osm:n1027979508": ("Oostende", ""),
    "osm:n26446051": ("Verviers-Central", ""),
    # --- NL ---
    "osm:n4085675596": ("'s-Hertogenbosch", ""),
    "osm:n4085675598": ("Alkmaar", ""),
    "osm:n4530820010": ("Almelo", ""),
    "osm:n4555468696": ("Almere Centrum", ""),
    "osm:n44727803": ("Arnhem Centraal", ""),
    "osm:n7606786768": ("Assen", ""),
    "osm:n43174364": ("Breda", ""),
    "osm:n4425618606": ("Dordrecht", ""),
    "osm:n4487559980": ("Enschede", ""),
    "osm:n1112410297": ("Groningen", ""),
    "osm:n5252716645": ("Haarlem", ""),
    "osm:n45931397": ("Hengelo", ""),
    "osm:n48162487": ("Leeuwarden", ""),
    "osm:n9604567339": ("Leiden Centraal", ""),
    "osm:n46792197": ("Lelystad Centrum", ""),
    "osm:n5311118145": ("Maastricht", ""),
    "osm:n44061300": ("Nijmegen", ""),
    "osm:n42966392": ("Roosendaal", ""),
    "osm:n4041466061": ("Tilburg", ""),
    "osm:n4555433902": ("Venlo", ""),
    "osm:n4487554970": ("Zwolle", ""),
}

In [4]:
# Iberia
ADDITIONS_IBERIA = {
    # --- ES ---
    "osm:n13782316672": ("A Coruña", ""),
    "osm:w28776478": ("Abando Indalecio Prieto", ""),
    "osm:n10914769161": ("Alacant Terminal", ""),
    "osm:n11016395831": ("Albacete Los Llanos", ""),
    "osm:n2182333421": ("Algeciras-Paco de Lucía", ""),
    "osm:n7499028215": ("Atocha-Cercanías", ""),
    "osm:n30546837": ("Avilés", ""),
    "osm:n2962633346": ("Badajoz", ""),
    "osm:n617134268": ("Cartagena", ""),
    "osm:n13894649638": ("Castelló", ""),
    "osm:n13717016710": ("Ciudad Real", ""),
    "osm:n13714976346": ("Cáceres", ""),
    "osm:n259625422": ("Cádiz", ""),
    "osm:n7567516121": ("Córdoba Julio Anguita", ""),
    "osm:n1738646773": ("El Puerto de Santa María", ""),
    "osm:w24930154": ("Ferrol", ""),
    "osm:n7201979115": ("Granada", ""),
    "osm:n6313228293": ("Guadalajara", ""),
    "osm:n5580567331": ("Huelva", ""),
    "osm:n8051540821": ("Huesca", ""),
    "osm:n7747872372": ("Huércal-Viator", ""),
    "osm:n5299078295": ("León", ""),
    "osm:n2459459539": ("Logroño", ""),
    "osm:w86123754": ("Lugo", ""),
    "osm:n13017334754": ("Murcia del Carmen", ""),
    "osm:n2609534280": ("Málaga María Zambrano", ""),
    "osm:n2039781019": ("Mérida", ""),
    "osm:n1842017741": ("Ourense-Empalme", ""),
    "osm:n4586092220": ("Oviedo / Uviéu", ""),
    "osm:n1939943099": ("Palencia", ""),
    "osm:n11757382798": ("Pamplona / Iruña", ""),
    "osm:n1069592576": ("Ponferrada", ""),
    "osm:n453651815": ("Pontevedra", ""),
    "osm:n4448930346": ("Salamanca", ""),
    "osm:n2340360836": ("San Fernando-Bahía Sur", ""),
    "osm:n5893228216": ("Santander", ""),
    "osm:n12768629140": ("Santiago de Compostela - Daniel Castelao", ""),
    "osm:n191262271": ("Sevilla - Santa Justa", ""),
    "osm:n2328131203": ("Talavera de la Reina", ""),
    "osm:n13894696022": ("Tarragona", ""),
    "osm:n7246471728": ("València - Estació del Nord", ""),
    "osm:n7246471727": ("València Joaquín Sorolla", ""),
    "osm:n12819737430": ("Vigo-Urzáiz", ""),
    "osm:n29568804": ("Vitoria-Gasteiz", ""),
    "osm:n791063229": ("Zamora", ""),
    # --- PT ---
    "osm:n10783341030": ("Aveiro", ""),
    "osm:n1393070418": ("Barroselas", ""),
    "osm:n10783341036": ("Braga", ""),
    "osm:n10783341029": ("Coimbra-B", ""),
    "osm:n46756926": ("Faro", ""),
    "osm:n10783341033": ("Gaia", ""),
    "osm:n3941781457": ("Guimarães", ""),
    "osm:n6297158592": ("Lisboa - Oriente", ""),
    "osm:n10783341023": ("Porto - Campanhã", ""),
    "osm:n2861321998": ("Viana do Castelo", ""),
}

In [5]:
# Italy
ADDITIONS_ITALY = {
    # --- IT ---
    "osm:n5324492776": ("Acireale", ""),
    "osm:n12291596709": ("Alessandria", ""),
    "osm:n7460292092": ("Ancona", ""),
    "osm:n593748428": ("Arezzo Pescaiola", ""),
    "osm:n8820637017": ("Avellino", ""),
    "osm:n1699232800": ("Barletta", ""),
    "osm:n8607336257": ("Bergamo", ""),
    "osm:n7473059189": ("Campobasso", ""),
    "osm:n1215079672": ("Caserta", ""),
    "osm:n603353943": ("Cefalù", ""),
    "osm:n258613100": ("Cerignola Campagna", ""),
    "osm:n1279764780": ("Cosenza Vaglio Lise", ""),
    "osm:n2121919441": ("Ferrara", ""),
    "osm:n738159083": ("L'Aquila", ""),
    "osm:n7042610811": ("Milazzo", ""),
    "osm:n726611782": ("Modena", ""),
    "osm:n11802851419": ("Novara", ""),
    "osm:n5836604869": ("Parma", ""),
    "osm:n211030020": ("Pavia", ""),
    "osm:n249236145": ("Perugia", ""),
    "osm:n1862274592": ("Pesaro", ""),
    "osm:n267591085": ("Pescara Centrale", ""),
    "osm:n13742535643": ("Piacenza", ""),
    "osm:n1670700542": ("Pordenone", ""),
    "osm:n842367835": ("Potenza Centrale", ""),
    "osm:n9067692438": ("Ravenna", ""),
    "osm:n82549162": ("Reggio Emilia", ""),
    "osm:n251338211": ("Rimini Torre Pedrera", ""),
    "osm:n1274172585": ("Sant'Agata di Militello", ""),
    "osm:n3395226356": ("Torino Porta Nuova", ""),
    "osm:n1764381735": ("Trento", ""),
    "osm:n3102397464": ("Vignale-Riotorto", ""),
}

In [6]:
# United Kingdom, Ireland
ADDITIONS_UNITED_KINGDOM = {
    # --- GB ---
    "osm:n7998566986": ("Ashford International", ""),
    "osm:n4461326005": ("Bangor", ""),
    "osm:n12248421687": ("Belfast Grand Central", ""),
    "osm:n6765532062": ("Birmingham New Street", ""),
    "osm:n6634567434": ("Bournemouth", ""),
    "osm:n7209380367": ("Bradford Interchange", ""),
    "osm:n20947173": ("Brighton", ""),
    "osm:n7167271113": ("Bristol Temple Meads", ""),
    "osm:n573566827": ("Cambridge", ""),
    "osm:n3453612249": ("Canterbury West", ""),
    "osm:n6605149666": ("Cardiff Central", ""),
    "osm:n5028607042": ("Chester", ""),
    "osm:n6688385690": ("Dover Priory", ""),
    "osm:n3663368461": ("Euston", ""),
    "osm:n6013523209": ("Exeter St Davids", ""),
    "osm:n6646199707": ("Gloucester", ""),
    "osm:n7154209250": ("Holyhead", ""),
    "osm:n6012826246": ("Hull Paragon Interchange", ""),
    "osm:n119274464": ("King's Cross St Pancras", ""),
    "osm:n7156706693": ("Leeds", ""),
    "osm:n4292139459": ("Leicester", ""),
    "osm:n6960405293": ("Liverpool Lime Street", ""),
    "osm:n5064005964": ("Manchester Piccadilly", ""),
    "osm:n7159380475": ("Milton Keynes Central", ""),
    "osm:n195885858": ("Newcastle", ""),
    "osm:n7158616254": ("Norwich", ""),
    "osm:n324650068": ("Nottingham", ""),
    "osm:n6481707942": ("Oxford", ""),
    "osm:n2612643529": ("Peterborough", ""),
    "osm:n6010790017": ("Portsmouth and Southsea", ""),
    "osm:n5784212748": ("Sheffield", ""),
    "osm:n638908005": ("Southampton Central", ""),
    "osm:n7140234411": ("Southend Victoria", ""),
    "osm:n6900337987": ("Swansea", ""),
    "osm:n104734": ("Swindon", ""),
    "osm:n6634567442": ("Wolverhampton", ""),
    "osm:n7989407332": ("Wrexham General", ""),
    # --- IE ---
    "osm:n5355226792": ("Cork Kent", ""),
    "osm:n7198337980": ("Dublin Connolly", ""),
    "osm:n6854406241": ("Dublin Heuston", ""),
    "osm:n6854415949": ("Galway Ceannt", ""),
    "osm:n6740364330": ("Limerick Colbert", ""),
    "osm:n6852246817": ("Waterford Plunkett", ""),
}

In [7]:
# Nordics
ADDITIONS_NORDICS = {
    # --- SE ---
    "osm:n662932765": ("Arlanda central", ""),
    "osm:n7135739559": ("Borås C", ""),
    "osm:r10274650": ("Härnösand resecentrum", ""),
    "osm:w419690632": ("Hässleholm C", ""),
    "osm:w155603926": ("Station Åre", ""),
    # --- DK ---
    "osm:n3419486092": ("Aalborg", ""),
    "osm:n10241003997": ("Aarhus H", ""),
    "osm:n1655765253": ("Hirtshals", ""),
    # --- FI ---
    "osm:n1716259527": ("Espoo", ""),
    "osm:n259004650": ("Jyväskylä", ""),
    "osm:n603918145": ("Kotka satama", ""),
    "osm:n292809487": ("Kuopio", ""),
    "osm:n537913195": ("Lahti", ""),
    "osm:n340019021": ("Tikkurila", ""),
    "osm:n91925127": ("Vaasa", ""),
    "osm:n4993961319": ("Esbjerg", "ferry:Esbjerg — carried over from the curated catalog, which step 7 now replaces"),
}

In [8]:
# Central Europe
ADDITIONS_CENTRAL_EUROPE = {
    # --- PL ---
    "osm:n3437310369": ("Bydgoszcz Główna", ""),
    "osm:n413346673": ("Elbląg", ""),
    "osm:n2050000245": ("Gorzów Wielkopolski", ""),
    "osm:n3831584572": ("Grudziądz", ""),
    "osm:n3357286930": ("Głogów", ""),
    "osm:n811380395": ("Inowrocław", ""),
    "osm:n3459064660": ("Kalisz", ""),
    "osm:n29830753": ("Konin", ""),
    "osm:n2627870779": ("Legnica", ""),
    "osm:n5356872372": ("Lubin", ""),
    "osm:n475592274": ("Nowy Sącz", ""),
    "osm:n131851982": ("Olsztyn Główny", ""),
    "osm:n528526130": ("Ostrów Wielkopolski", ""),
    "osm:n1864585459": ("Piotrków Trybunalski", ""),
    "osm:n3459469757": ("Piła Główna", ""),
    "osm:n1991734621": ("Płock", ""),
    "osm:n842079165": ("Wałbrzych Miasto", ""),
    "osm:n3459316528": ("Włocławek", ""),
    "osm:n372254443": ("Zamość", ""),
    "osm:n1992495561": ("Łomża", ""),
    # --- CZ ---
    "osm:n3279883031": ("Hradec Králové hlavní nádraží", ""),
    "osm:n5648124921": ("Most", ""),
    "osm:n30077682": ("Plzeň hlavní nádraží", ""),
    # --- SK ---
    "osm:n8012637130": ("Banská Bystrica", ""),
    "osm:n7066101885": ("Nitra", ""),
    "osm:n346420319": ("Prešov", ""),
    "osm:n1911997558": ("Trenčín", ""),
    "osm:n10607895201": ("Zvolen nákladná stanica", ""),
    "osm:n6446509215": ("Žilina", ""),
    # --- HU ---
    "osm:n5217774800": ("Déli pályaudvar", ""),
    "osm:n5217774781": ("Kelenföld vasútállomás", ""),
    "osm:n268213797": ("Miskolc-Tiszai", ""),
    "osm:n25546152": ("Pécs", ""),
    "osm:n93800956": ("Szombathely", ""),
    "osm:n5217774764": ("Vörösmarty utca", ""),
    "osm:n3129289404": ("Česká Třebová", "network — junction on the Praha–Brno/Vienna corridor"),
    "osm:n24684084": ("Kolín", "network — junction on the Praha–Brno/Vienna corridor"),
}

In [9]:
# Baltics
ADDITIONS_BALTICS = {
    # --- EE ---
    "osm:n529932898": ("Narva", ""),
    "osm:n30402685": ("Paldiski", ""),
    "osm:n8761131036": ("Tartu", ""),
    # --- LV ---
    "osm:n7800844381": ("Daugavpils", ""),
    "osm:n7799314251": ("Liepāja", ""),
    "osm:n252636397": ("Ventspils-1", ""),
    # --- LT ---
    "osm:n99172829": ("Klaipėda", ""),
    "osm:n6624987344": ("Vilnius", ""),
    "osm:n1370770391": ("Šiauliai", ""),
}

In [10]:
# South-eastern Europe
ADDITIONS_SOUTH_EASTERN_EUROPE = {
    # --- SI ---
    "osm:n283207146": ("Bled Jezero", ""),
    "osm:n270129111": ("Postojna", ""),
    # --- HR ---
    "osm:n5668860678": ("Pula", ""),
    # --- BA ---
    "osm:n2038791178": ("Doboj", ""),
    "osm:n312280159": ("Mostar", ""),
    "osm:n942406094": ("Sarajevo", ""),
    "osm:n10212563520": ("Zenica", ""),
    # --- RS ---
    "osm:n1601596124": ("Ниш", ""),
    "osm:n893439322": ("Суботица", ""),
    # --- ME ---
    "osm:n10728069934": ("Nikšić", ""),
    # --- MK ---
    "osm:n3407148607": ("Битола", ""),
    "osm:n1635043504": ("Велес", ""),
    "osm:n134701987": ("Гевгелија", ""),
    "osm:n408145528": ("Куманово", ""),
    "osm:n9947846021": ("Скопје", ""),
    # --- AL ---
    "osm:n13895194677": ("Durrës", ""),
    "osm:n13895194676": ("Terminali i Transportit Publik Tiranë", ""),
    # --- XK ---
    "osm:n2107256271": ("Ferizaj", ""),
    "osm:n1613271652": ("Prishtinë", ""),
    # --- BG ---
    "osm:w529540422": ("Димитровград", ""),
    "osm:w421067799": ("Силистра", ""),
    "osm:n14055863804": ("Сливен", ""),
    "osm:n13254223966": ("Централна гара Русе", ""),
    "osm:n13254223964": ("Централна гара София", ""),
    # --- RO ---
    "osm:n1243304538": ("Botoșani", ""),
    "osm:n9244693114": ("Brăila", ""),
    "osm:n487098710": ("Călărași Sud", ""),
    "osm:w272187224": ("Dej Triaj", ""),
    "osm:n345539748": ("Focșani", ""),
    "osm:n8176378047": ("Galați", ""),
    "osm:n247416268": ("Iași", "night_train_stop (step 6 had Nicolina, a secondary station)"),
    "osm:n303278705": ("Piatra-Neamț", ""),
    "osm:n8607130227": ("Pitești", ""),
    "osm:n516125655": ("Râmnicu Vâlcea", ""),
    "osm:n202022357": ("Sighișoara", ""),
    "osm:n536752658": ("Slatina", ""),
    "osm:n7905078406": ("Tulcea Oraș", ""),
    "osm:n9554314257": ("Târgoviște", ""),
    "osm:n13577341022": ("Târgu Jiu", ""),
    "osm:n258668249": ("Târgu Mureș", ""),
    # --- GR ---
    "osm:n9643537166": ("Βόλος", ""),
    "osm:n4925130296": ("Θεσσαλονίκη", ""),
    "osm:n6175162599": ("Κατερίνη", ""),
    "osm:n287554537": ("Λάρισα", ""),
    "osm:n308841776": ("Ξάνθη", ""),
    "osm:n9721698903": ("Ρίο", ""),
    "osm:n4883927548": ("Σέρραι", ""),
    "osm:n251083102": ("Σταθμός Λαρίσης", ""),
    "osm:n13254223965": ("Централна гара Бургас", "night_train_stop (absent from ONTD)"),
    "osm:w216482601": ("Велико Търново", "night_train_stop (Горна Оряховица is the regional hub, but the schedule calls at Veliko Tarnovo)"),
}

In [11]:
# Eastern Europe, Türkiye
ADDITIONS_EASTERN_EUROPE = {
    # --- UA ---
    "osm:n9708140929": ("Алчевськ", ""),
    "osm:n3815614789": ("Бердянськ (експ.)", ""),
    "osm:n3376816024": ("Вокзальна", ""),
    "osm:n9695166282": ("Дебальцеве", ""),
    "osm:n278290881": ("Донецьк", ""),
    "osm:n676673228": ("Евпатория-Курорт", ""),
    "osm:n8594603350": ("Житомир", ""),
    "osm:n2469903575": ("Керчь-Порт", ""),
    "osm:n10175634575": ("Луганськ", ""),
    "osm:n1201406760": ("Маріуполь", ""),
    "osm:n652003827": ("Мелітополь", ""),
    "osm:n2635702849": ("Миколаїв", "night_train_stop (step 6 had Миколаїв-Вантажний, a freight station)"),
    "osm:n8220188327": ("Росинка", ""),
    "osm:n305264647": ("Севастополь", ""),
    "osm:n3693422903": ("Симферополь", ""),
    "osm:n11742271914": ("Херсон", ""),
    "osm:n4237097872": ("Шепетівка", ""),
    # --- TR ---
    "osm:n2726063373": ("Adana", ""),
    "osm:n1033799242": ("Denizli", ""),
    "osm:w88493479": ("Gaziantep Garı", ""),
    "osm:n1035592982": ("Isparta", ""),
    "osm:n5014804022": ("Konya", ""),
    "osm:n11082593543": ("Kırkikievler", ""),
    "osm:n2596542075": ("Mersin Garı", ""),
    "osm:n1550726413": ("Muş", ""),
    "osm:n125326513": ("Osmaniye", ""),
    "osm:n1023854236": ("Samsun", ""),
    "osm:n13480008012": ("Zonguldak", ""),
    "osm:n4896717721": ("tren", ""),
    "osm:n719596870": ("Полтава-Київська", "night_train_stop (absent from ONTD) — 19 trips"),
    "osm:n1276842137": ("Кривий Ріг-Головний", "night_train_stop (replaces Кривий Ріг, a smaller station 3.8 km off)"),
}

## Combine, resolve and write

In [12]:
ADDITIONS = {}
for group in (
    ADDITIONS_GERMANY,
    ADDITIONS_FRANCE,
    ADDITIONS_IBERIA,
    ADDITIONS_ITALY,
    ADDITIONS_UNITED_KINGDOM,
    ADDITIONS_NORDICS,
    ADDITIONS_CENTRAL_EUROPE,
    ADDITIONS_BALTICS,
    ADDITIONS_SOUTH_EASTERN_EUROPE,
    ADDITIONS_EASTERN_EUROPE,
):
    overlap = ADDITIONS.keys() & group.keys()
    if overlap:
        raise ValueError(f"stop listed in two regions: {sorted(overlap)}")
    ADDITIONS.update(group)

print(f"manual additions: {len(ADDITIONS)}")

manual additions: 404


In [13]:
# Name and coordinates come from step 3b. Country prefers ONTD via step 4
# (curated national data); step 3b's own country column is only ~1% populated,
# so the handful of stops ONTD doesn't cover are listed explicitly below rather
# than pulled from the legacy step 6 file — twelve values are not worth a file
# dependency, and here they are visible and reviewable.
MANUAL_COUNTRY = {
    "osm:w529540422": "BG",
    "osm:w421067799": "BG",
    "osm:n14055863804": "BG",
    "osm:w28776478": "ES",
    "osm:w24930154": "ES",
    "osm:w86123754": "ES",
    "osm:w73575850": "FR",
    "osm:w112095036": "FR",
    "osm:w272187224": "RO",
    "osm:n202022357": "RO",
    "osm:r10274650": "SE",
    "osm:n1550726413": "TR",
    "osm:n247416268": "RO",
    "osm:w216482601": "BG",
    "osm:n4993961319": "DK",
    "osm:n60093107": "AT",
    "osm:n3129289404": "CZ",
    "osm:n24684084": "CZ",
}
osm = {}
with open(ensure_local("step3b_output_osm_stations_classified.csv"), encoding="utf-8-sig", newline="") as fh:
    for row in csv.DictReader(fh):
        osm[row["stop_id"]] = row

ontd_country = {}
with open(ensure_local("step4_MatchingONTDtoOSM.csv"), encoding="utf-8-sig", newline="") as fh:
    for row in csv.DictReader(fh):
        stop_id = (row.get("osm_stop_id") or "").strip()
        country = (row.get("ontd_country") or "").strip().upper()
        if stop_id and country:
            ontd_country.setdefault(stop_id, country)

unknown = sorted(set(ADDITIONS) - set(osm))
if unknown:
    raise KeyError(
        f"{len(unknown)} stop id(s) not in step 3b — typo, or the OSM extract was "
        f"refreshed and the object is gone: {unknown[:10]}"
    )

rows = []
for stop_id, (label, reason) in ADDITIONS.items():
    station = osm[stop_id]
    rows.append(
        {
            "stop_id": stop_id,
            "stop_name": station["stop_name"].strip() or label,
            "country": (
                ontd_country.get(stop_id)
                or station["country"].strip().upper()
                or MANUAL_COUNTRY.get(stop_id, "")
            ),
            "stop_lat": station["stop_lat"],
            "stop_lon": station["stop_lon"],
            "reason": reason.strip(),
        }
    )
rows.sort(key=lambda r: (r["country"], r["stop_name"]))

with open(OUTPUT_PATH, "w", encoding="utf-8-sig", newline="") as fh:
    writer = csv.DictWriter(
        fh, fieldnames=["stop_id", "stop_name", "country", "stop_lat", "stop_lon", "reason"]
    )
    writer.writeheader()
    writer.writerows(rows)

print(f"wrote {len(rows)} rows to {OUTPUT_PATH.name}")
print("by country:", Counter(r["country"] for r in rows).most_common(8))

# ONTD overrides the country the legacy step 6 file carried for these stops.
# Recorded so the change stays visible rather than silently altering a country —
# notably the Crimean stations, which the legacy file coded RU against ONTD's UA.
LEGACY_COUNTRY_OVERRIDDEN = {
    "osm:n529932898": ("Narva", "RU"),
    "osm:n5893228216": ("Santander", "NO"),
    "osm:n676673228": ("Евпатория-Курорт", "RU"),
    "osm:n2469903575": ("Керчь-Порт", "RU"),
    "osm:n305264647": ("Севастополь", "RU"),
    "osm:n3693422903": ("Симферополь", "RU"),
}
corrections = [
    (r["stop_id"], r["stop_name"], LEGACY_COUNTRY_OVERRIDDEN[r["stop_id"]][1], r["country"])
    for r in rows
    if r["stop_id"] in LEGACY_COUNTRY_OVERRIDDEN
    and LEGACY_COUNTRY_OVERRIDDEN[r["stop_id"]][1] != r["country"]
]
if corrections:
    print(f"\ncountry corrected from ONTD on {len(corrections)} stops:")
    for stop_id, name, was, now in corrections:
        print(f"  {name[:38]:40} {was} -> {now}  ({stop_id})")

no_country = [r for r in rows if not r["country"]]
if no_country:
    raise ValueError(
        f"{len(no_country)} addition(s) have no country — step 7 cannot derive a "
        f"timezone and would drop them: "
        f"{[(r['stop_id'], r['stop_name']) for r in no_country]}. "
        "Add them to MANUAL_COUNTRY above."
    )

wrote 404 rows to step6_manual_additions.csv
by country: [('FR', 48), ('DE', 45), ('ES', 45), ('GB', 37), ('IT', 32), ('NL', 21), ('PL', 20), ('UA', 19)]

country corrected from ONTD on 6 stops:
  Narva                                    RU -> EE  (osm:n529932898)
  Santander                                NO -> ES  (osm:n5893228216)
  Евпатория-Курорт                         RU -> UA  (osm:n676673228)
  Керчь-Порт                               RU -> UA  (osm:n2469903575)
  Севастополь                              RU -> UA  (osm:n305264647)
  Симферополь                              RU -> UA  (osm:n3693422903)


## Unfilled reasons

Everything listed here is a stop in the public catalog that cannot yet explain
why it is there.

In [14]:
missing = [r for r in rows if not r["reason"]]
print(f"{len(missing)} of {len(rows)} additions have no reason recorded\n")
for row in missing:
    print(f"  {row['country']}  {row['stop_name'][:44]:46} {row['stop_id']}")

394 of 404 additions have no reason recorded

  AL  Durrës                                         osm:n13895194677
  AL  Terminali i Transportit Publik Tiranë          osm:n13895194676
  BA  Doboj                                          osm:n2038791178
  BA  Mostar                                         osm:n312280159
  BA  Sarajevo                                       osm:n942406094
  BA  Zenica                                         osm:n10212563520
  BE  Brugge                                         osm:n2929614444
  BE  Diamant                                        osm:n12095463891
  BE  Gare du Midi - Zuidstation                     osm:n4878787366
  BE  Gent-Sint-Pieters                              osm:n1178257779
  BE  Kortrijk                                       osm:n21309047
  BE  La Louvière-Centre                             osm:n446059037
  BE  Mechelen-Nekkerspoel                           osm:n7261826908
  BE  Oostende                                       osm:n